<a href="https://colab.research.google.com/github/TetianaMar-888/Child-internet-use-datamining/blob/main/notebooks/DM2_04_feature_fusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import gzip, pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA = '/content/drive/MyDrive/2_PROJECTS/DM2_25_26/data'
REPORTS = '/content/drive/MyDrive/2_PROJECTS/DM2_25_26/reports'

# --- tabular, with the Module 0/1 corrections reapplied ---
data = pd.read_csv(f'{DATA}/cmi_internet.csv')

cgas_mask = data['CGAS-CGAS_Score'] > 100
data.loc[cgas_mask, 'CGAS-CGAS_Score'] = data.loc[cgas_mask, 'CGAS-CGAS_Score'] / 10

for col in ['Physical-BMI', 'Physical-Weight', 'Physical-Diastolic_BP', 'Physical-Systolic_BP']:
    data[col] = data[col].replace(0, np.nan)

data['BIA-BIA_Fat'] = data['BIA-BIA_Fat'].where(data['BIA-BIA_Fat'].between(0, 100))

bia_numeric = [c for c in data.columns if c.startswith('BIA-BIA_')]
for col in bia_numeric:
    q99 = data[col].quantile(0.99)
    data.loc[data[col] > q99 * 3, col] = np.nan

print("Tabular:", data.shape)

# --- time series ---
with gzip.open(f'{DATA}/CMI_timeseries_dataset.pkl.gz', 'rb') as f:
    ts_dataset = pickle.load(f)

ts_ids_all = np.array([df['id'].iloc[0] for df in ts_dataset])
print("Time series:", len(ts_dataset), "windows from", len(set(ts_ids_all)), "children")

# --- the critical question: how do the id spaces relate? ---
print("\nTabular id dtype:", data['id'].dtype, "| sample:", data['id'].head(3).tolist())
print("Time-series id dtype:", ts_ids_all.dtype, "| sample:", ts_ids_all[:3].tolist())

overlap = set(data['id'].astype(str)) & set(pd.Series(ts_ids_all).astype(str))
print(f"\nIds present in both datasets: {len(overlap)}")

Mounted at /content/drive
Tabular: (8460, 82)
Time series: 4437 windows from 494 children

Tabular id dtype: int64 | sample: [0, 1, 2]
Time-series id dtype: int64 | sample: [2265, 51, 3042]

Ids present in both datasets: 494


In [4]:
SIGNAL_COLS = ['X', 'Y', 'Z', 'enmo', 'anglez', 'light']
ts_array = np.stack([df[SIGNAL_COLS].values for df in ts_dataset]).astype(np.float64)

# Module 0 corrections: spike removal
for idx in [0, 1, 2, 3]:
    w, t = np.where(np.abs(ts_array[:, :, idx]) > 3)
    for wi, ti in zip(w, t):
        lo, hi = max(0, ti-1), min(199, ti+1)
        ts_array[wi, ti, idx] = (ts_array[wi, lo, idx] + ts_array[wi, hi, idx]) / 2

nonwear = np.stack([df['non-wear_flag'].values for df in ts_dataset]).astype(np.float64)
worn = nonwear < 0.5

enmo = ts_array[:, :, SIGNAL_COLS.index('enmo')]
anglez = ts_array[:, :, SIGNAL_COLS.index('anglez')]
light = ts_array[:, :, SIGNAL_COLS.index('light')]

rows = []
for i in range(len(ts_dataset)):
    m = worn[i]
    if m.sum() < 20:          # too little worn time to summarise
        continue
    e, a, l = enmo[i][m], anglez[i][m], light[i][m]

    rows.append({
        'id': ts_ids_all[i],
        # activity magnitude
        'ts_enmo_mean': e.mean(),
        'ts_enmo_std': e.std(),
        'ts_enmo_p90': np.percentile(e, 90),
        'ts_enmo_active_frac': (e > 0.1).mean(),      # share of time in moderate activity
        'ts_enmo_still_frac': (e < 0.01).mean(),      # share of time near-motionless
        # posture / sleep proxy
        'ts_anglez_std': a.std(),
        'ts_anglez_stable_frac': (np.abs(np.diff(a)) < 5).mean(),  # postural stability
        # light exposure
        'ts_light_mean_log': np.log1p(l).mean(),
        'ts_light_dark_frac': (l < 1).mean(),
        # compliance
        'ts_nonwear_frac': 1 - m.mean(),
    })

window_feat = pd.DataFrame(rows)
print(f"Windows summarised: {len(window_feat)} of {len(ts_dataset)}")

# Aggregate windows to one row per child
ts_features = window_feat.groupby('id').mean()
ts_features['ts_n_windows'] = window_feat.groupby('id').size()
print(f"Children with time-series features: {len(ts_features)}")
print(ts_features.describe().round(4).T)

Windows summarised: 4391 of 4437
Children with time-series features: 451
                       count     mean      std     min      25%      50%  \
ts_enmo_mean           451.0   0.0451   0.0274  0.0000   0.0246   0.0464   
ts_enmo_std            451.0   0.0552   0.0380  0.0000   0.0261   0.0544   
ts_enmo_p90            451.0   0.1048   0.0710  0.0000   0.0534   0.1043   
ts_enmo_active_frac    451.0   0.1132   0.0900  0.0000   0.0264   0.1116   
ts_enmo_still_frac     451.0   0.3033   0.2775  0.0000   0.1296   0.1790   
ts_anglez_std          451.0  26.6186  11.9672  0.0060  22.2050  24.8555   
ts_anglez_stable_frac  451.0   0.4482   0.2459  0.2094   0.2859   0.3193   
ts_light_mean_log      451.0   2.3064   0.7465  0.0000   2.0344   2.3653   
ts_light_dark_frac     451.0   0.1164   0.2086  0.0000   0.0215   0.0587   
ts_nonwear_frac        451.0   0.1607   0.2914  0.0000   0.0000   0.0000   
ts_n_windows           451.0   9.7361  10.6720  1.0000   1.0000   3.0000   

              

In [5]:
# Children with zero recorded activity despite the device being worn — real or artefact?
suspicious = ts_features[ts_features['ts_enmo_mean'] < 0.001]
print(f"Children with near-zero mean activity: {len(suspicious)}")
print(suspicious[['ts_enmo_mean', 'ts_enmo_still_frac', 'ts_anglez_std',
                  'ts_nonwear_frac', 'ts_n_windows']].round(4))

Children with near-zero mean activity: 22
      ts_enmo_mean  ts_enmo_still_frac  ts_anglez_std  ts_nonwear_frac  \
id                                                                       
310         0.0007              0.9744        41.1328            0.805   
325         0.0003              0.9902        29.4622            0.490   
575         0.0000              1.0000         0.4033            0.895   
599         0.0000              1.0000         0.1073            0.785   
739         0.0000              1.0000         0.3165            0.895   
847         0.0000              1.0000         0.6975            0.790   
861         0.0007              1.0000         0.0146            0.730   
989         0.0000              1.0000         0.0635            0.835   
1303        0.0000              1.0000         0.0598            0.865   
2238        0.0000              1.0000         0.0060            0.875   
2251        0.0000              1.0000         0.0145            0.610

In [6]:
data['target_binary'] = (data['sii'] > 0).astype(int)

merged = data.merge(ts_features, left_on='id', right_index=True, how='inner')
print(f"Merged dataset: {merged.shape[0]} children, {merged.shape[1]} columns")
print(f"Class balance: {merged['target_binary'].value_counts(normalize=True).round(3).to_dict()}")
print(f"(full tabular set was 0.689 / 0.311)")

# Do the new features correlate with the target at all?
ts_cols = [c for c in merged.columns if c.startswith('ts_')]
corr_ts = merged[ts_cols + ['target_binary']].corr()['target_binary'].drop('target_binary')
print("\nCorrelation with binary target:")
print(corr_ts.sort_values(key=abs, ascending=False).round(3))

Merged dataset: 451 children, 94 columns
Class balance: {0: 0.623, 1: 0.377}
(full tabular set was 0.689 / 0.311)

Correlation with binary target:
ts_enmo_active_frac     -0.206
ts_enmo_mean            -0.186
ts_enmo_p90             -0.156
ts_enmo_std             -0.154
ts_n_windows            -0.085
ts_anglez_stable_frac    0.067
ts_light_dark_frac       0.067
ts_light_mean_log       -0.066
ts_enmo_still_frac       0.066
ts_nonwear_frac          0.046
ts_anglez_std            0.040
Name: target_binary, dtype: float64


In [7]:
# Tighten the threshold: require at least half the window to be genuinely worn
rows = []
for i in range(len(ts_dataset)):
    m = worn[i]
    if m.sum() < 100:            # was 20 — too permissive
        continue
    e, a, l = enmo[i][m], anglez[i][m], light[i][m]
    rows.append({
        'id': ts_ids_all[i],
        'ts_enmo_mean': e.mean(), 'ts_enmo_std': e.std(),
        'ts_enmo_p90': np.percentile(e, 90),
        'ts_enmo_active_frac': (e > 0.1).mean(),
        'ts_enmo_still_frac': (e < 0.01).mean(),
        'ts_anglez_std': a.std(),
        'ts_anglez_stable_frac': (np.abs(np.diff(a)) < 5).mean(),
        'ts_light_mean_log': np.log1p(l).mean(),
        'ts_light_dark_frac': (l < 1).mean(),
        'ts_nonwear_frac': 1 - m.mean(),
    })

window_feat = pd.DataFrame(rows)
ts_features = window_feat.groupby('id').mean()
ts_features['ts_n_windows'] = window_feat.groupby('id').size()

print(f"Windows retained: {len(window_feat)} of {len(ts_dataset)}")
print(f"Children: {len(ts_features)}")
print(f"Children with near-zero activity now: {(ts_features['ts_enmo_mean'] < 0.001).sum()}")

Windows retained: 4308 of 4437
Children: 373
Children with near-zero activity now: 3


In [8]:
still_left = ts_features[ts_features['ts_enmo_mean'] < 0.001]
print(still_left[['ts_enmo_mean', 'ts_enmo_still_frac', 'ts_anglez_std',
                  'ts_nonwear_frac', 'ts_n_windows']].round(4))

      ts_enmo_mean  ts_enmo_still_frac  ts_anglez_std  ts_nonwear_frac  \
id                                                                       
325         0.0003              0.9902        29.4622            0.490   
2482        0.0000              1.0000         2.0676            0.415   
2496        0.0000              1.0000        65.2186            0.310   

      ts_n_windows  
id                  
325              1  
2482             1  
2496             1  


In [9]:
merged = data.merge(ts_features, left_on='id', right_index=True, how='inner')
print(f"Merged: {merged.shape[0]} children")
print(f"Class balance: {merged['target_binary'].value_counts(normalize=True).round(3).to_dict()}")

ts_cols = [c for c in merged.columns if c.startswith('ts_')]
corr_ts = merged[ts_cols + ['target_binary']].corr()['target_binary'].drop('target_binary')
print("\nCorrelation with binary target (cleaned features):")
print(corr_ts.sort_values(key=abs, ascending=False).round(3))

Merged: 373 children
Class balance: {0: 0.63, 1: 0.37}

Correlation with binary target (cleaned features):
ts_enmo_active_frac     -0.246
ts_enmo_mean            -0.239
ts_enmo_p90             -0.191
ts_enmo_std             -0.178
ts_enmo_still_frac       0.137
ts_light_mean_log       -0.109
ts_n_windows            -0.088
ts_anglez_stable_frac    0.070
ts_anglez_std            0.065
ts_light_dark_frac       0.057
ts_nonwear_frac          0.017
Name: target_binary, dtype: float64


In [10]:
ts_features = ts_features[ts_features['ts_enmo_mean'] >= 0.001]
merged = data.merge(ts_features, left_on='id', right_index=True, how='inner')
print(f"Final merged dataset: {merged.shape[0]} children")
print(f"Class balance: {merged['target_binary'].value_counts(normalize=True).round(3).to_dict()}")

Final merged dataset: 370 children
Class balance: {0: 0.63, 1: 0.37}


## Feature Fusion: combining the tabular and time-series datasets

The two datasets share a child identifier, and all 494 children with
accelerometer data appear in the tabular set. This raises a question neither
module addresses on its own: does objectively measured physical activity add
information beyond what the questionnaires and physical measurements already
provide?

### Extracting features from the raw signal

Ten features were derived per window and averaged to one row per child, grouped
into four families:

- **Activity magnitude** — mean ENMO, its standard deviation, the 90th
  percentile, the fraction of time in moderate activity (ENMO > 0.1) and the
  fraction near-motionless (ENMO < 0.01).
- **Posture** — Z-angle standard deviation and the fraction of consecutive
  steps with angular change under 5°, together acting as a crude sleep proxy.
- **Light exposure** — log-mean ambient light and the fraction of time in
  darkness.
- **Compliance** — the fraction of non-wear time, which Module 0 found to be
  statistically associated with the target.

All statistics are computed on worn time only. Including non-wear periods would
record a device on a table as perfect stillness.

### A threshold that mattered

An initial requirement of at least 20 worn steps per window (10% of a window)
produced 22 children with a mean ENMO of essentially zero across their entire
record. Inspection showed all 22 had exactly one window with non-wear fractions
between 0.31 and 0.895, and several had a Z-angle standard deviation below 0.1 —
a device held at a perfectly constant angle, which no worn sensor produces.
These were statistics computed on a handful of surviving timesteps rather than
genuine measurements.

Raising the threshold to 100 worn steps (half a window) reduced the anomalies
from 22 to 3, which were then removed individually. The cost was substantial —
451 children fell to 370 — but the correction **strengthened** the signal rather
than weakening it: correlation of `ts_enmo_active_frac` with the target rose from
−0.206 to −0.246, and `ts_enmo_mean` from −0.186 to −0.239. The artefactual
children had been diluting a real relationship.

### The merged dataset

370 children, 32 tabular features plus 11 derived from the accelerometer. The
class balance shifts from 0.689 / 0.311 in the full tabular set to 0.630 / 0.370
here: children with accelerometer data are somewhat more likely to be in the
problematic group, so results generalise to this subset rather than to the
population.

The strongest new features correlate with the binary target at −0.246
(`ts_enmo_active_frac`) and −0.239 (`ts_enmo_mean`) — comparable to the best
tabular predictors, and in the expected direction: less movement, more
problematic use.

In [11]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

tabular_features = [
    'Basic_Demos-Age', 'Basic_Demos-Sex', 'CGAS-CGAS_Score',
    'Physical-BMI', 'Physical-Height', 'Physical-Weight',
    'Physical-Waist_Circumference', 'Physical-Diastolic_BP',
    'Physical-Systolic_BP', 'Physical-HeartRate',
    'Fitness_Endurance-Max_Stage', 'Fitness_Endurance-Time_Mins',
    'FGC-FGC_CU', 'FGC-FGC_GSND', 'FGC-FGC_GSD', 'FGC-FGC_PU',
    'FGC-FGC_SRL', 'FGC-FGC_SRR', 'FGC-FGC_TL',
    'BIA-BIA_BMI', 'BIA-BIA_Fat', 'BIA-BIA_FFMI', 'BIA-BIA_SMM',
    'BIA-BIA_TBW', 'BIA-BIA_ICW', 'BIA-BIA_ECW', 'BIA-BIA_BMR', 'BIA-BIA_DEE',
    'PAQ_A-PAQ_A_Total', 'PAQ_C-PAQ_C_Total', 'SDS-SDS_Total_Raw',
    'PreInt_EduHx-computerinternet_hoursday',
]
tabular_features = [c for c in tabular_features if c in merged.columns]
ts_cols = [c for c in merged.columns if c.startswith('ts_')]

y = merged['target_binary']
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

configs = {
    'Tabular only': tabular_features,
    'Time-series only': ts_cols,
    'Tabular + time-series': tabular_features + ts_cols,
}

for name, feats in configs.items():
    pipe = make_pipeline(
        SimpleImputer(strategy='median'),
        RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_leaf=10,
                               class_weight='balanced', random_state=42, n_jobs=-1)
    )
    auc = cross_val_score(pipe, merged[feats], y, cv=cv, scoring='roc_auc')
    f1 = cross_val_score(pipe, merged[feats], y, cv=cv, scoring='f1_macro')
    print(f"{name:24s} n_feat={len(feats):3d}  "
          f"AUC {auc.mean():.3f}±{auc.std():.3f}   macro-F1 {f1.mean():.3f}±{f1.std():.3f}")

Tabular only             n_feat= 32  AUC 0.775±0.044   macro-F1 0.705±0.044
Time-series only         n_feat= 11  AUC 0.644±0.062   macro-F1 0.623±0.061
Tabular + time-series    n_feat= 43  AUC 0.778±0.054   macro-F1 0.680±0.047


In [12]:
# Are the time-series features just re-measuring what the tabular ones already capture?
tab_activity = ['PAQ_A-PAQ_A_Total', 'PAQ_C-PAQ_C_Total',
                'Fitness_Endurance-Max_Stage', 'Fitness_Endurance-Time_Mins',
                'FGC-FGC_CU', 'FGC-FGC_PU', 'Basic_Demos-Age', 'Physical-BMI']
tab_activity = [c for c in tab_activity if c in merged.columns]

cross = merged[ts_cols + tab_activity].corr().loc[ts_cols, tab_activity]
print("Strongest correlations between time-series and tabular features:")
strong = cross.abs().stack().sort_values(ascending=False).head(12)
for (ts_f, tab_f), v in strong.items():
    print(f"  {ts_f:24s} ↔ {tab_f:30s} {cross.loc[ts_f, tab_f]:+.3f}")

Strongest correlations between time-series and tabular features:
  ts_enmo_active_frac      ↔ Basic_Demos-Age                -0.445
  ts_enmo_mean             ↔ Basic_Demos-Age                -0.435
  ts_enmo_std              ↔ Basic_Demos-Age                -0.391
  ts_enmo_p90              ↔ Basic_Demos-Age                -0.390
  ts_enmo_still_frac       ↔ Basic_Demos-Age                +0.258
  ts_enmo_std              ↔ Physical-BMI                   -0.253
  ts_enmo_mean             ↔ Physical-BMI                   -0.212
  ts_enmo_p90              ↔ Physical-BMI                   -0.205
  ts_enmo_active_frac      ↔ Physical-BMI                   -0.200
  ts_enmo_p90              ↔ PAQ_C-PAQ_C_Total              +0.194
  ts_enmo_std              ↔ PAQ_C-PAQ_C_Total              +0.193
  ts_enmo_active_frac      ↔ PAQ_C-PAQ_C_Total              +0.181


In [13]:
pipe_full = make_pipeline(
    SimpleImputer(strategy='median'),
    RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_leaf=10,
                           class_weight='balanced', random_state=42, n_jobs=-1)
)
pipe_full.fit(merged[tabular_features + ts_cols], y)
rf = pipe_full.named_steps['randomforestclassifier']

imp = pd.Series(rf.feature_importances_, index=tabular_features + ts_cols).sort_values(ascending=False)
print("\nTop 15 features in the combined model:")
for f, v in imp.head(15).items():
    marker = "  ← time series" if f.startswith('ts_') else ""
    print(f"  {f:40s} {v:.4f}{marker}")
print(f"\nTime-series features in top 15: {sum(1 for f in imp.head(15).index if f.startswith('ts_'))} of 11")


Top 15 features in the combined model:
  Physical-Height                          0.0799
  Physical-Weight                          0.0698
  Basic_Demos-Age                          0.0670
  SDS-SDS_Total_Raw                        0.0627
  PreInt_EduHx-computerinternet_hoursday   0.0555
  BIA-BIA_BMR                              0.0510
  BIA-BIA_SMM                              0.0494
  ts_enmo_mean                             0.0486  ← time series
  BIA-BIA_ICW                              0.0333
  ts_enmo_active_frac                      0.0321  ← time series
  ts_enmo_p90                              0.0286  ← time series
  ts_enmo_std                              0.0261  ← time series
  ts_enmo_still_frac                       0.0260  ← time series
  FGC-FGC_PU                               0.0242
  BIA-BIA_TBW                              0.0218

Time-series features in top 15: 5 of 11


In [14]:
from scipy import stats

def partial_corr(df, x, y, control):
    d = df[[x, y, control]].dropna()
    rx = d[x] - (stats.linregress(d[control], d[x]).slope * d[control]
                 + stats.linregress(d[control], d[x]).intercept)
    ry = d[y] - (stats.linregress(d[control], d[y]).slope * d[control]
                 + stats.linregress(d[control], d[y]).intercept)
    return np.corrcoef(rx, ry)[0, 1]

print("Correlation with target, before and after controlling for age:")
for f in ['ts_enmo_active_frac', 'ts_enmo_mean', 'ts_enmo_p90', 'ts_enmo_std']:
    raw = merged[[f, 'target_binary']].corr().iloc[0, 1]
    pc = partial_corr(merged, f, 'target_binary', 'Basic_Demos-Age')
    print(f"  {f:24s} {raw:+.3f} → {pc:+.3f}")

Correlation with target, before and after controlling for age:
  ts_enmo_active_frac      -0.250 → -0.099
  ts_enmo_mean             -0.246 → -0.098
  ts_enmo_p90              -0.196 → -0.057
  ts_enmo_std              -0.182 → -0.041


### Does the accelerometer add anything?

The same Random Forest (300 trees, depth 10, minimum 10 samples per leaf,
balanced class weights) was evaluated on three feature sets under identical
5-fold stratified cross-validation. Cross-validation rather than a single split,
because 370 records make any one split too unstable to interpret.

| Feature set | Features | ROC-AUC | Macro F1 |
|---|---|---|---|
| Tabular only | 32 | 0.775 ± 0.044 | **0.705 ± 0.044** |
| Time-series only | 11 | 0.644 ± 0.062 | 0.623 ± 0.061 |
| Tabular + time-series | 43 | **0.778 ± 0.054** | 0.680 ± 0.047 |

**Adding the accelerometer features changes nothing.** AUC rises by 0.003 — an
eighth of one standard deviation — and macro F1 *falls* by 0.025. The
time-series features alone reach 0.644, well below the 0.775 from tabular data,
so they are neither redundant-but-equal nor complementary: they are weaker and
they overlap.

Note that the tabular-only figure of 0.775 is not comparable to the 0.704 macro
AUC reported in Module 2. This is a binary task on a 370-child subset with a
different class balance, not the four-class task on 8,460 children.

### Why it fails: the sensor is largely measuring age

Cross-correlating the two feature sets identifies the overlap immediately:

| Time-series feature | Tabular feature | r |
|---|---|---|
| `ts_enmo_active_frac` | `Basic_Demos-Age` | **−0.445** |
| `ts_enmo_mean` | `Basic_Demos-Age` | **−0.435** |
| `ts_enmo_std` | `Basic_Demos-Age` | −0.391 |
| `ts_enmo_p90` | `Basic_Demos-Age` | −0.390 |
| `ts_enmo_std` | `Physical-BMI` | −0.253 |
| `ts_enmo_p90` | `PAQ_C-PAQ_C_Total` | +0.194 |

Age is the strongest correlate of every activity feature, and by a wide margin —
stronger than any of them correlates with the target. Younger children move
more than adolescents, so an accelerometer substantially records age, which the
model already knows from `Basic_Demos-Age`, `Physical-Height` and
`Physical-Weight` — the three features occupying the top three importance
positions.

Partial correlation confirms this quantitatively. Controlling for age removes
roughly 60% of each feature's association with the target:

| Feature | Raw r | Age-controlled r |
|---|---|---|
| `ts_enmo_active_frac` | −0.250 | −0.099 |
| `ts_enmo_mean` | −0.246 | −0.098 |
| `ts_enmo_p90` | −0.196 | −0.057 |
| `ts_enmo_std` | −0.182 | −0.041 |

This mirrors the finding in Module 0, where height and weight lost roughly half
their association with the target once age was controlled for. The same
mechanism operates here on different data.

A residual 40% survives, so objectively measured activity does carry a small
independent contribution — it is simply too small to move an AUC whose
cross-validated standard deviation is 0.044 on a sample of 370.

**Notably, the objective and self-reported measures of activity barely agree.**
`ts_enmo_p90` correlates with the children's activity questionnaire at only
0.194. A child's accelerometer-measured movement and their reported activity
level are nearly independent — which makes it more surprising, not less, that
neither improves on the other.

### A parting observation on feature importance

Five of the eleven time-series features enter the combined model's top 15, with
`ts_enmo_mean` ranked eighth at 0.0486 — above most tabular features. The model
demonstrably uses them, and the AUC does not move regardless.

This is a clean illustration of what impurity importance does and does not mean.
A Random Forest will readily split on `ts_enmo_mean` because it is a clean
continuous variable carrying genuine signal. But that signal is already
reachable through age and body size, so substituting one path for another
changes which features get used without changing what the model knows. High
importance indicates a feature is *useful for splitting*, not that it adds
*information*. Section 3.8 of the report makes the same point from the opposite
direction, where SHAP revealed a feature that impurity importance had
understated.

### Conclusion

Fusing the two datasets does not improve prediction. The accelerometer's
contribution is real but largely redundant with age, and the sample of 370
children — the price of requiring both data sources and adequate wear time — is
too small for a 40% residual effect to register.

The experiment is nonetheless worth reporting. It establishes that the ceiling
documented throughout this project is not an artefact of relying on
questionnaires and physical measurements: adding an objective behavioural
measurement from an entirely different instrument does not raise it either.